In [ ]:
# ============================================================
# CELL 1 — UPLOAD AUTHORITATIVE STORY OUTLINE
#
# Upload exactly one UTF-8 .txt outline.
# Story language is detected from the uploaded text itself.
# No Google Drive.
# Everything for this run stays under /content.
# ============================================================

from google.colab import files
from IPython.display import clear_output
from pathlib import Path
from datetime import datetime

import importlib.util
import re
import subprocess
import sys


# ------------------------------------------------------------
# A. FIXED PIPELINE CONSTRAINTS
# ------------------------------------------------------------

STORY_MIN_WORDS = 1300
STORY_MAX_WORDS = 1500
STORY_TARGET_WORDS = 1450

PROJECT_ROOT = Path(
    "/content/audio_story_factory"
)

RUNS_ROOT = (
    PROJECT_ROOT
    / "runs"
)

RUNS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# B. UPLOAD OUTLINE
# ------------------------------------------------------------

print(
    "Upload exactly one .txt story outline."
)

uploaded = files.upload()

clear_output(
    wait=True
)

if len(uploaded) != 1:
    raise RuntimeError(
        f"Expected exactly one uploaded file; "
        f"received {len(uploaded)}."
    )

OUTLINE_FILENAME, OUTLINE_BYTES = (
    next(
        iter(
            uploaded.items()
        )
    )
)

if not (
    OUTLINE_FILENAME
    .lower()
    .endswith(".txt")
):
    raise RuntimeError(
        "Expected a .txt outline, received: "
        f"{OUTLINE_FILENAME}"
    )


# ------------------------------------------------------------
# C. DECODE INPUT
# ------------------------------------------------------------

try:

    STORY_OUTLINE = (
        OUTLINE_BYTES
        .decode(
            "utf-8-sig"
        )
    )

except UnicodeDecodeError as exc:

    raise RuntimeError(
        "The outline must be UTF-8 text."
    ) from exc


STORY_OUTLINE = (
    STORY_OUTLINE.strip()
)

if not STORY_OUTLINE:
    raise RuntimeError(
        "The uploaded outline is empty."
    )


# ------------------------------------------------------------
# D. DETECT LANGUAGE FROM THE ACTUAL OUTLINE
#
# Nothing about the story language is supplied manually.
# langdetect reads the uploaded text.
# ------------------------------------------------------------

if (
    importlib.util.find_spec(
        "langdetect"
    )
    is None
):

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "langdetect",
        ],
        check=True,
    )


from langdetect import (
    detect_langs,
    DetectorFactory,
)


DetectorFactory.seed = 0


try:

    LANGUAGE_RESULTS = (
        detect_langs(
            STORY_OUTLINE
        )
    )

except Exception as exc:

    raise RuntimeError(
        "Could not determine the language "
        "of the uploaded outline."
    ) from exc


if not LANGUAGE_RESULTS:

    raise RuntimeError(
        "Language detector returned no result."
    )


BEST_LANGUAGE = (
    LANGUAGE_RESULTS[0]
)

STORY_LANGUAGE_CODE = (
    BEST_LANGUAGE.lang
    .lower()
)


# langdetect distinguishes simplified/traditional Chinese.
# Qwen uses a single Chinese language entry.
if (
    STORY_LANGUAGE_CODE
    .startswith("zh")
):

    STORY_LANGUAGE_CODE = "zh"


LANGUAGE_NAMES = {
    "en": "English",
    "zh": "Chinese",
    "ja": "Japanese",
    "ko": "Korean",
    "de": "German",
    "fr": "French",
    "ru": "Russian",
    "pt": "Portuguese",
    "es": "Spanish",
    "it": "Italian",
}


if (
    STORY_LANGUAGE_CODE
    not in LANGUAGE_NAMES
):

    raise RuntimeError(
        "The uploaded outline was detected as "
        f"{BEST_LANGUAGE.lang!r}, which is not "
        "a language supported by the current "
        "Qwen3-TTS pipeline."
    )


STORY_LANGUAGE = (
    LANGUAGE_NAMES[
        STORY_LANGUAGE_CODE
    ]
)

STORY_LANGUAGE_CONFIDENCE = (
    float(
        BEST_LANGUAGE.prob
    )
)


# ------------------------------------------------------------
# E. CREATE THIS RUN'S DIRECTORIES
# ------------------------------------------------------------

OUTLINE_FILE_STEM = (
    Path(
        OUTLINE_FILENAME
    ).stem
)

SAFE_OUTLINE_STEM = (
    re.sub(
        r"[^A-Za-z0-9._-]+",
        "_",
        OUTLINE_FILE_STEM,
    )
    .strip(
        "._-"
    )
    or
    "story"
)

RUN_TIMESTAMP = (
    datetime.now()
    .strftime(
        "%Y%m%d_%H%M%S"
    )
)

RUN_ID = (
    f"{SAFE_OUTLINE_STEM}_"
    f"{RUN_TIMESTAMP}"
)

RUN_ROOT = (
    RUNS_ROOT
    / RUN_ID
)

INPUT_DIR = (
    RUN_ROOT
    / "input"
)

STORY_DIR = (
    RUN_ROOT
    / "story"
)

VERSIONS_DIR = (
    RUN_ROOT
    / "versions"
)

AUDITS_DIR = (
    RUN_ROOT
    / "audits"
)

REVISIONS_DIR = (
    RUN_ROOT
    / "revisions"
)

META_DIR = (
    RUN_ROOT
    / "meta"
)

AUDIO_DIR = (
    RUN_ROOT
    / "audio"
)

AUDIO_CHUNKS_DIR = (
    AUDIO_DIR
    / "chunks"
)


for directory in (
    INPUT_DIR,
    STORY_DIR,
    VERSIONS_DIR,
    AUDITS_DIR,
    REVISIONS_DIR,
    META_DIR,
    AUDIO_DIR,
    AUDIO_CHUNKS_DIR,
):

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ------------------------------------------------------------
# F. SAVE THE ORIGINAL INPUT
# ------------------------------------------------------------

OUTLINE_PATH = (
    INPUT_DIR
    / "authoritative_outline.txt"
)

OUTLINE_PATH.write_bytes(
    OUTLINE_BYTES
)


# ------------------------------------------------------------
# G. REPORT WHAT WAS ACTUALLY READ
# ------------------------------------------------------------

print(
    "Outline loaded successfully."
)

print(
    "File:",
    OUTLINE_FILENAME,
)

print(
    "Run:",
    RUN_ID,
)

print(
    "Outline characters:",
    len(
        STORY_OUTLINE
    ),
)

print(
    "Detected language:",
    STORY_LANGUAGE,
    f"({STORY_LANGUAGE_CODE})",
)

print(
    "Detection confidence:",
    f"{STORY_LANGUAGE_CONFIDENCE:.4f}",
)

print(
    "Strict story length:",
    f"{STORY_MIN_WORDS}-"
    f"{STORY_MAX_WORDS} words",
    "| target:",
    STORY_TARGET_WORDS,
)

Outline loaded successfully.
File: Romantic.txt
Run: Romantic_20260914_110112
Outline characters: 1637
Detected language: English (en)
Detection confidence: 1.0000
Strict story length: 1300-1500 words | target: 1450


In [ ]:
# ============================================================
# CELL 2 — INSTALL, DOWNLOAD, LOAD ALL THREE MODELS INTO CPU RAM
#
# Full BF16 models. NO quantization.
#
# Writer: Qwen/Qwen3-32B
# Judge : deepseek-ai/DeepSeek-R1-Distill-Qwen-32B
# TTS   : Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice
#
# End state:
#   WRITER_LLM  -> CPU RAM
#   JUDGE_LLM   -> CPU RAM
#   TTS_ENGINE  -> CPU RAM
#   GPU          -> empty
#
# Cell 3 moves exactly one text model at a time CPU <-> A100.
# Cell 4 moves TTS to GPU only after the storyline is fixed.
# No Google Drive.
# ============================================================

import os
import sys
import gc
import shutil
import subprocess
import time
from pathlib import Path

# ------------------------------------------------------------
# A. CLEAN OLD RUNTIME OBJECTS, BUT PRESERVE CELL 1 STATE
# ------------------------------------------------------------

for name in (
    "WRITER_LLM",
    "WRITER_TOKENIZER",
    "JUDGE_LLM",
    "JUDGE_TOKENIZER",
    "TTS_ENGINE",
):
    if name in globals():
        try:
            obj = globals().pop(name)
            del obj
        except Exception:
            pass

gc.collect()

try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass
except Exception:
    pass

# Remove model artifacts from the older quantized notebook if they exist.
# The new full-BF16 model root is separate and is reused if Cell 2 is rerun.
OLD_MODEL_ROOT = Path("/content/audio_story_models")
OLD_HF_CACHE = Path("/content/hf_cache")

for stale in (OLD_MODEL_ROOT, OLD_HF_CACHE):
    if stale.exists():
        shutil.rmtree(stale, ignore_errors=True)

MODEL_ROOT = Path("/content/audio_story_models_full_bf16")
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = "/content/hf_home_audio_story"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ.pop("HF_HUB_OFFLINE", None)
os.environ.pop("TRANSFORMERS_OFFLINE", None)

# ------------------------------------------------------------
# B. INSTALL SOFTWARE
# ------------------------------------------------------------

print("=" * 76)
print("INSTALLING RUNTIME")
print("=" * 76, flush=True)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "--upgrade",
        "transformers==4.57.3",
        "accelerate==1.12.0",
        "huggingface_hub[hf_xet]",
        "safetensors",
        "sentencepiece",
        "psutil",
        "qwen-tts==0.1.1",
        "soundfile",
    ],
    check=True,
)

import psutil
import torch
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer, AutoModelForCausalLM
from qwen_tts import Qwen3TTSModel

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required.")

gpu = torch.cuda.get_device_properties(0)
gpu_total_gib = gpu.total_memory / 1024**3

print("\nGPU:", gpu.name)
print(f"GPU memory: {gpu_total_gib:.2f} GiB")

if gpu_total_gib < 75:
    raise RuntimeError(
        "This notebook is designed for an A100 80 GB-class GPU. "
        f"Detected only {gpu_total_gib:.2f} GiB."
    )

vm = psutil.virtual_memory()
disk = shutil.disk_usage("/content")

print(
    "CPU RAM:",
    f"{vm.total / 1024**3:.2f} GiB total | "
    f"{vm.available / 1024**3:.2f} GiB available",
)
print(
    "Local disk:",
    f"{disk.total / 1024**3:.2f} GiB total | "
    f"{disk.free / 1024**3:.2f} GiB free",
)

# Both 32B BF16 models plus TTS must coexist in CPU RAM.
if vm.available / 1024**3 < 135:
    raise RuntimeError(
        "Insufficient currently available CPU RAM for both full BF16 "
        "32B LLMs plus TTS. Need roughly 135+ GiB available before loading."
    )

# ------------------------------------------------------------
# C. MODEL IDENTITIES
# ------------------------------------------------------------

WRITER_REPO = "Qwen/Qwen3-32B"
JUDGE_REPO = "deepseek-ai/DeepSeek-R1-Distill-Qwen-32B"
TTS_REPO = "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice"

WRITER_DIR = MODEL_ROOT / "Qwen3-32B"
JUDGE_DIR = MODEL_ROOT / "DeepSeek-R1-Distill-Qwen-32B"
TTS_MODEL_DIR = MODEL_ROOT / "Qwen3-TTS-12Hz-1.7B-CustomVoice"

# ------------------------------------------------------------
# D. DOWNLOAD ALL THREE MODELS TO LOCAL /content
#
# snapshot_download(local_dir=...) keeps one local copy per model.
# No Google Drive and no quantized files.
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("1/3 — DOWNLOADING FULL QWEN3-32B WRITER")
print("=" * 76, flush=True)

snapshot_download(
    repo_id=WRITER_REPO,
    local_dir=str(WRITER_DIR),
    max_workers=4,
)

print("\n" + "=" * 76)
print("2/3 — DOWNLOADING FULL DEEPSEEK-R1-DISTILL-QWEN-32B JUDGE")
print("=" * 76, flush=True)

snapshot_download(
    repo_id=JUDGE_REPO,
    local_dir=str(JUDGE_DIR),
    max_workers=4,
)

print("\n" + "=" * 76)
print("3/3 — DOWNLOADING QWEN3-TTS")
print("=" * 76, flush=True)

snapshot_download(
    repo_id=TTS_REPO,
    local_dir=str(TTS_MODEL_DIR),
    max_workers=4,
)

def verify_transformer_snapshot(path, label):
    path = Path(path)
    if not (path / "config.json").is_file():
        raise RuntimeError(f"{label}: config.json missing.")
    if not list(path.glob("*.safetensors")):
        raise RuntimeError(f"{label}: no safetensors weights found.")

verify_transformer_snapshot(WRITER_DIR, "Writer")
verify_transformer_snapshot(JUDGE_DIR, "Judge")

if not (TTS_MODEL_DIR / "model.safetensors").is_file():
    raise RuntimeError("Qwen3-TTS main model.safetensors is missing.")

if not (
    TTS_MODEL_DIR
    / "speech_tokenizer"
    / "model.safetensors"
).is_file():
    raise RuntimeError("Qwen3-TTS speech tokenizer weights are missing.")

print("\nAll model downloads verified.", flush=True)

# ------------------------------------------------------------
# E. LOAD FULL QWEN3-32B WRITER INTO CPU RAM
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("LOADING QWEN3-32B WRITER INTO CPU RAM — BF16, NO QUANTIZATION")
print("=" * 76, flush=True)

WRITER_TOKENIZER = AutoTokenizer.from_pretrained(
    str(WRITER_DIR),
    local_files_only=True,
    trust_remote_code=True,
)

if WRITER_TOKENIZER.pad_token_id is None:
    WRITER_TOKENIZER.pad_token = WRITER_TOKENIZER.eos_token

WRITER_LLM = AutoModelForCausalLM.from_pretrained(
    str(WRITER_DIR),
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    attn_implementation="sdpa",
    local_files_only=True,
    trust_remote_code=True,
)

WRITER_LLM.eval()

if next(WRITER_LLM.parameters()).device.type != "cpu":
    raise RuntimeError("Writer was expected in CPU RAM after Cell 2.")

print("Writer resident in CPU RAM.", flush=True)

# ------------------------------------------------------------
# F. LOAD FULL DEEPSEEK 32B JUDGE INTO CPU RAM
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("LOADING DEEPSEEK-R1-DISTILL-QWEN-32B INTO CPU RAM — BF16")
print("=" * 76, flush=True)

JUDGE_TOKENIZER = AutoTokenizer.from_pretrained(
    str(JUDGE_DIR),
    local_files_only=True,
    trust_remote_code=True,
)

if JUDGE_TOKENIZER.pad_token_id is None:
    JUDGE_TOKENIZER.pad_token = JUDGE_TOKENIZER.eos_token

JUDGE_LLM = AutoModelForCausalLM.from_pretrained(
    str(JUDGE_DIR),
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    attn_implementation="sdpa",
    local_files_only=True,
    trust_remote_code=True,
)

JUDGE_LLM.eval()

if next(JUDGE_LLM.parameters()).device.type != "cpu":
    raise RuntimeError("Judge was expected in CPU RAM after Cell 2.")

print("Judge resident in CPU RAM.", flush=True)

# ------------------------------------------------------------
# G. LOAD QWEN3-TTS INTO CPU RAM
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("LOADING QWEN3-TTS INTO CPU RAM")
print("=" * 76, flush=True)

TTS_ENGINE = Qwen3TTSModel.from_pretrained(
    str(TTS_MODEL_DIR),
    dtype=torch.bfloat16,
    local_files_only=True,
)

TTS_DEVICE = next(
    TTS_ENGINE.model.parameters()
).device

if TTS_DEVICE.type != "cpu":
    raise RuntimeError(
        f"TTS was expected in CPU RAM, got {TTS_DEVICE}."
    )

TTS_SPEAKERS = TTS_ENGINE.get_supported_speakers() or []
TTS_LANGUAGES = TTS_ENGINE.get_supported_languages() or []

# ------------------------------------------------------------
# H. VERIFY FINAL RESIDENCY
# ------------------------------------------------------------

for label, model in (
    ("Writer", WRITER_LLM),
    ("Judge", JUDGE_LLM),
):
    device = next(model.parameters()).device
    if device.type != "cpu":
        raise RuntimeError(f"{label} is not resident in CPU RAM: {device}")

if next(TTS_ENGINE.model.parameters()).device.type != "cpu":
    raise RuntimeError("TTS is not resident in CPU RAM.")

gc.collect()
torch.cuda.empty_cache()

try:
    torch.cuda.ipc_collect()
except Exception:
    pass

free_vram, total_vram = torch.cuda.mem_get_info()

process_rss = psutil.Process(
    os.getpid()
).memory_info().rss / 1024**3

print("\n" + "=" * 76)
print("CELL 2 COMPLETE — ALL MODELS RESIDENT IN CPU RAM")
print("=" * 76)
print("Writer:", WRITER_REPO, "| BF16 | CPU")
print("Judge :", JUDGE_REPO, "| BF16 | CPU")
print("TTS   :", TTS_REPO, "| BF16 | CPU")
print(f"Python process RSS: {process_rss:.2f} GiB")
print(
    "GPU free:",
    f"{free_vram / 1024**3:.2f} / "
    f"{total_vram / 1024**3:.2f} GiB",
)
print("TTS speakers:", TTS_SPEAKERS)
print("TTS languages:", TTS_LANGUAGES)


INSTALLING RUNTIME



    If you do not have SoX, proceed here:
     - - - http://sox.sourceforge.net/ - - -

    If you do (or think that you should) have SoX, double-check your
    path variables.
    



********
********
 

GPU: NVIDIA A100-SXM4-80GB
GPU memory: 79.25 GiB
CPU RAM: 167.05 GiB total | 162.04 GiB available
Local disk: 235.68 GiB total | 187.97 GiB free

1/3 — DOWNLOADING FULL QWEN3-32B WRITER


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 27 files:   0%|          | 0/27 [00:00<?, ?it/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/17 [00:00<?, ?it/s]

Writer resident in CPU RAM.

LOADING DEEPSEEK-R1-DISTILL-QWEN-32B INTO CPU RAM — BF16


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

Judge resident in CPU RAM.

LOADING QWEN3-TTS INTO CPU RAM

CELL 2 COMPLETE — ALL MODELS RESIDENT IN CPU RAM
Writer: Qwen/Qwen3-32B | BF16 | CPU
Judge : deepseek-ai/DeepSeek-R1-Distill-Qwen-32B | BF16 | CPU
TTS   : Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice | BF16 | CPU
Python process RSS: 3.89 GiB
GPU free: 78.83 / 79.25 GiB
TTS speakers: ['aiden', 'dylan', 'eric', 'ono_anna', 'ryan', 'serena', 'sohee', 'uncle_fu', 'vivian']
TTS languages: ['auto', 'chinese', 'english', 'french', 'german', 'italian', 'japanese', 'korean', 'portuguese', 'russian', 'spanish']


In [ ]:
# ============================================================
# CELL 3 — STRICT LOCAL STORY PIPELINE
# ============================================================

import gc
import json
import re
import time
import unicodedata
from pathlib import Path

import torch
from transformers import StoppingCriteria, StoppingCriteriaList

REQUIRED = (
    "STORY_OUTLINE",
    "STORY_LANGUAGE",
    "STORY_MIN_WORDS",
    "STORY_MAX_WORDS",
    "STORY_TARGET_WORDS",
    "WRITER_LLM",
    "WRITER_TOKENIZER",
    "JUDGE_LLM",
    "JUDGE_TOKENIZER",
    "TTS_ENGINE",
    "STORY_DIR",
    "VERSIONS_DIR",
    "AUDITS_DIR",
    "REVISIONS_DIR",
    "META_DIR",
)

missing = [
    x
    for x in REQUIRED
    if x not in globals()
]

if missing:
    raise RuntimeError(
        "Run Cells 1 and 2 first. Missing: "
        + ", ".join(missing)
    )


# ------------------------------------------------------------
# Story generation
#
# IMPORTANT:
# max_new_tokens is now only a large safety ceiling.
# Actual stopping is controlled by Python WORD COUNT.
# ------------------------------------------------------------

STORY_MIN_NEW_TOKENS = 3000
STORY_MAX_NEW_TOKENS = 5000

MAX_LENGTH_REVISIONS = 2

JUDGE_MAX_NEW_TOKENS = 8192

REPAIR_MAX_NEW_TOKENS = 4096
MAX_REPAIR_PASSES = 3

PIPELINE_START = time.perf_counter()


for directory in (
    STORY_DIR,
    VERSIONS_DIR,
    AUDITS_DIR,
    REVISIONS_DIR,
    META_DIR,
):
    Path(directory).mkdir(
        parents=True,
        exist_ok=True,
    )


# ============================================================
# HELPERS
# ============================================================

def save(path, text):
    Path(path).write_text(
        str(text),
        encoding="utf-8",
    )


def save_json(path, obj):
    save(
        path,
        json.dumps(
            obj,
            ensure_ascii=False,
            indent=2,
        ),
    )


def wc(text):
    return len(
        re.findall(
            r"\S+",
            str(text).strip(),
        )
    )


def split_story(text):
    return [
        paragraph.strip()
        for paragraph in re.split(
            r"\n\s*\n",
            str(text).strip(),
        )
        if paragraph.strip()
    ]


def join_story(paragraphs):
    return "\n\n".join(
        paragraphs
    )


def numbered(paragraphs):
    return "\n\n".join(
        f"[P{i:03d}] {paragraph}"
        for i, paragraph
        in enumerate(
            paragraphs,
            1,
        )
    )


_TRANS = str.maketrans({
    "“": '"',
    "”": '"',
    "„": '"',
    "‟": '"',
    "‘": "'",
    "’": "'",
    "‚": "'",
    "‛": "'",
    "–": "-",
    "—": "-",
    "−": "-",
    "…": "...",
    "\u00a0": " ",
})


def norm(text):

    if text is None:
        return ""

    text = unicodedata.normalize(
        "NFKC",
        str(text),
    ).translate(
        _TRANS
    )

    return re.sub(
        r"\s+",
        " ",
        text,
    ).strip().casefold()


def occurs(anchor, source):

    return (
        bool(
            norm(anchor)
        )
        and
        norm(anchor)
        in norm(source)
    )


def pid(
    value,
    allow_none=False,
):

    value = (
        ""
        if value is None
        else str(value).strip()
    )

    if (
        allow_none
        and
        value.upper()
        in {
            "",
            "NONE",
            "NULL",
        }
    ):
        return None

    match = re.fullmatch(
        r"(?i)P\s*0*(\d+)",
        value,
    )

    return (
        int(
            match.group(1)
        )
        if match
        else -1
    )


def optional_text(value):

    value = (
        ""
        if value is None
        else str(value).strip()
    )

    return (
        None
        if norm(value)
        in {
            "",
            "none",
            "null",
        }
        else value
    )


def clean_writer_text(text):

    text = str(
        text
    ).strip()

    text = re.sub(
        r"^\s*```(?:text|markdown)?\s*",
        "",
        text,
        flags=re.I,
    )

    text = re.sub(
        r"\s*```\s*$",
        "",
        text,
    ).strip()

    text = re.sub(
        r"<think>.*?</think>",
        "",
        text,
        flags=re.S | re.I,
    ).strip()

    return text


def extract_json_dict(
    text,
    required_keys=(),
):

    decoder = json.JSONDecoder()

    hits = []

    text = str(
        text
    )

    for start, character in enumerate(
        text
    ):

        if character != "{":
            continue

        try:
            obj, used = decoder.raw_decode(
                text[
                    start:
                ]
            )

        except json.JSONDecodeError:
            continue

        if (
            isinstance(
                obj,
                dict,
            )
            and
            all(
                key in obj
                for key in required_keys
            )
        ):
            hits.append(
                (
                    start + used,
                    obj,
                )
            )

    if not hits:
        raise ValueError(
            "No valid JSON object containing "
            f"{required_keys} found."
        )

    return max(
        hits,
        key=lambda item:
            item[
                0
            ],
    )[
        1
    ]


# ============================================================
# CPU <-> GPU SWAPPING
# ============================================================

def model_device(model):
    return next(
        model.parameters()
    ).device


def gpu_report(
    prefix="GPU residency:",
):

    free_vram, total_vram = (
        torch.cuda.mem_get_info()
    )

    print(
        prefix,
        f"{(total_vram - free_vram) / 1024**3:.2f} / "
        f"{total_vram / 1024**3:.2f} GiB GPU used",
        flush=True,
    )


def move_to_cpu(
    model,
    label,
):

    if model_device(
        model
    ).type == "cpu":
        return

    print(
        f"Swapping {label}: GPU -> CPU ...",
        flush=True,
    )

    started = time.perf_counter()

    model.to(
        "cpu"
    )

    gc.collect()

    torch.cuda.empty_cache()

    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass

    print(
        f"{label} now in CPU RAM "
        f"({time.perf_counter() - started:.1f}s).",
        flush=True,
    )


def move_to_gpu(
    model,
    label,
):

    if model_device(
        model
    ).type == "cuda":
        return

    print(
        f"Swapping {label}: CPU -> A100 ...",
        flush=True,
    )

    started = time.perf_counter()

    model.to(
        device="cuda:0",
        dtype=torch.bfloat16,
    )

    model.eval()

    torch.cuda.synchronize()

    print(
        f"{label} now on A100 "
        f"({time.perf_counter() - started:.1f}s).",
        flush=True,
    )

    gpu_report()


def activate_writer():

    move_to_cpu(
        JUDGE_LLM,
        "Judge",
    )

    move_to_gpu(
        WRITER_LLM,
        "Writer",
    )


def activate_judge():

    move_to_cpu(
        WRITER_LLM,
        "Writer",
    )

    move_to_gpu(
        JUDGE_LLM,
        "Judge",
    )


def park_text_models():

    move_to_cpu(
        WRITER_LLM,
        "Writer",
    )

    move_to_cpu(
        JUDGE_LLM,
        "Judge",
    )

    gc.collect()

    torch.cuda.empty_cache()

    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass


# ============================================================
# CHAT / GENERATION HELPERS
# ============================================================

def render_chat(
    tokenizer,
    messages,
    enable_thinking=None,
):

    kwargs = {
        "tokenize":
            False,

        "add_generation_prompt":
            True,
    }

    if enable_thinking is not None:
        kwargs[
            "enable_thinking"
        ] = enable_thinking

    try:

        return tokenizer.apply_chat_template(
            messages,
            **kwargs,
        )

    except TypeError:

        kwargs.pop(
            "enable_thinking",
            None,
        )

        return tokenizer.apply_chat_template(
            messages,
            **kwargs,
        )


def context_limit(model):

    value = getattr(
        model.config,
        "max_position_embeddings",
        None,
    )

    if (
        isinstance(
            value,
            int,
        )
        and
        value > 0
    ):
        return value

    return 32768


# ============================================================
# WORD-COUNT STOPPING
#
# This is the critical correction.
#
# We DO NOT guess that 3400 tokens == enough story.
# Python watches the ACTUAL generated word count.
#
# At target length it stops at the first sentence ending.
# ============================================================

class StoryWordStopper(
    StoppingCriteria
):

    def __init__(
        self,
        tokenizer,
        prompt_tokens,
        target_words,
        max_words,
    ):

        self.tokenizer = tokenizer

        self.prompt_tokens = int(
            prompt_tokens
        )

        self.target_words = int(
            target_words
        )

        self.max_words = int(
            max_words
        )

        self.last_words = 0


    def __call__(
        self,
        input_ids,
        scores,
        **kwargs,
    ):

        generated_tokens = int(
            input_ids.shape[
                -1
            ]
            -
            self.prompt_tokens
        )

        # No reason to repeatedly decode very early output.
        if generated_tokens < 1800:
            return False

        # Until reasonably near the requested range,
        # inspect every 16 tokens.
        if (
            self.last_words
            <
            self.target_words
            -
            120
            and
            generated_tokens
            %
            16
            !=
            0
        ):
            return False

        generated = (
            self.tokenizer.decode(
                input_ids[
                    0,
                    self.prompt_tokens:
                ],
                skip_special_tokens=True,
            )
            .strip()
        )

        words = wc(
            generated
        )

        self.last_words = words

        # Preferred stop:
        # at/after target and on a sentence boundary.
        if words >= self.target_words:

            if re.search(
                r'[.!?…]["”»’\)\]]*$',
                generated,
            ):
                return True

        # Absolute word ceiling.
        return (
            words
            >=
            self.max_words
        )


def generate_local(
    label,
    model,
    tokenizer,
    messages,
    *,
    min_new_tokens=0,
    max_new_tokens,
    temperature,
    top_p,
    repetition_penalty=1.0,
    enable_thinking=None,
    word_stop=False,
):

    if model_device(
        model
    ).type != "cuda":

        raise RuntimeError(
            f"{label}: model is not on GPU."
        )

    prompt = render_chat(
        tokenizer,
        messages,
        enable_thinking=
            enable_thinking,
    )

    encoded = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
    )

    input_tokens = int(
        encoded[
            "input_ids"
        ].shape[
            -1
        ]
    )

    limit = context_limit(
        model
    )

    if (
        input_tokens
        +
        max_new_tokens
        +
        64
        >
        limit
    ):

        raise RuntimeError(
            f"{label}: token budget exceeds context: "
            f"input={input_tokens}, "
            f"output_cap={max_new_tokens}, "
            f"context={limit}."
        )

    encoded = {
        key:
            value.to(
                "cuda:0"
            )

        for key, value
        in encoded.items()
    }

    pad_token_id = (
        tokenizer.pad_token_id

        if tokenizer.pad_token_id
        is not None

        else tokenizer.eos_token_id
    )

    stopping_criteria = None

    if word_stop:

        stopping_criteria = (
            StoppingCriteriaList(
                [
                    StoryWordStopper(
                        tokenizer=
                            tokenizer,

                        prompt_tokens=
                            input_tokens,

                        target_words=
                            STORY_TARGET_WORDS,

                        max_words=
                            STORY_MAX_WORDS,
                    )
                ]
            )
        )

    generation_kwargs = dict(
        **encoded,

        min_new_tokens=int(
            min_new_tokens
        ),

        max_new_tokens=int(
            max_new_tokens
        ),

        do_sample=True,

        temperature=float(
            temperature
        ),

        top_p=float(
            top_p
        ),

        repetition_penalty=float(
            repetition_penalty
        ),

        use_cache=True,

        pad_token_id=
            pad_token_id,

        eos_token_id=
            tokenizer.eos_token_id,
    )

    if stopping_criteria is not None:

        generation_kwargs[
            "stopping_criteria"
        ] = stopping_criteria

    started = time.perf_counter()

    with torch.inference_mode():

        output_ids = model.generate(
            **generation_kwargs
        )

    generated_ids = output_ids[
        0,
        input_tokens:
    ]

    generated_tokens = int(
        generated_ids.numel()
    )

    text = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

    cleaned = clean_writer_text(
        text
    )

    hit_token_cap = (
        generated_tokens
        >=
        max_new_tokens
        -
        2
    )

    print(
        f"{label}: "
        f"input={input_tokens} tok | "
        f"output={generated_tokens} tok | "
        f"words={wc(cleaned)} | "
        f"{time.perf_counter() - started:.1f}s",
        flush=True,
    )

    del encoded
    del generation_kwargs
    del output_ids
    del generated_ids

    gc.collect()

    torch.cuda.empty_cache()

    return (
        text,
        generated_tokens,
        hit_token_cap,
    )


# ============================================================
# WRITER — STRICT 1300-1500
# ============================================================

WRITER_SYSTEM = f"""
You are the fiction WRITER.

Write the complete narration in {STORY_LANGUAGE}.

The user's outline is authoritative.

Preserve every explicit event, fact, relationship, chronology
requirement, setting, character, motivation and ending.

Where the outline is silent, invent only compatible material.

HARD LENGTH CONTRACT:

- final narration MUST contain between
  {STORY_MIN_WORDS} and {STORY_MAX_WORDS}
  whitespace-delimited words;

- target approximately {STORY_TARGET_WORDS} words;

- do not summarize;

- develop scenes fully;

- complete the entire ending before stopping.

Write natural prose with natural paragraph boundaries.

Return narration only.

No title unless explicitly required by the outline.
No headings.
No notes.
No analysis.
No planning.
No word-count commentary.
""".strip()


def writer_call(
    label,
    user_prompt,
    temperature,
):

    (
        raw,
        generated_tokens,
        hit_token_cap,
    ) = generate_local(
        label,

        WRITER_LLM,

        WRITER_TOKENIZER,

        [
            {
                "role":
                    "system",

                "content":
                    WRITER_SYSTEM,
            },

            {
                "role":
                    "user",

                "content":
                    user_prompt,
            },
        ],

        min_new_tokens=
            STORY_MIN_NEW_TOKENS,

        max_new_tokens=
            STORY_MAX_NEW_TOKENS,

        temperature=
            temperature,

        top_p=
            0.90,

        repetition_penalty=
            1.03,

        enable_thinking=
            False,

        word_stop=
            True,
    )

    return (
        clean_writer_text(
            raw
        ),
        generated_tokens,
        hit_token_cap,
    )


def writer_generate_initial():

    return writer_call(
        "writer_initial",

        (
            "<AUTHORITATIVE_OUTLINE>\n"
            f"{STORY_OUTLINE}\n"
            "</AUTHORITATIVE_OUTLINE>\n\n"

            f"Write the complete finished "
            f"{STORY_LANGUAGE} story now.\n"

            f"The legal final interval is "
            f"{STORY_MIN_WORDS}-"
            f"{STORY_MAX_WORDS} words.\n"

            f"Aim near "
            f"{STORY_TARGET_WORDS} words."
        ),

        0.76,
    )


def writer_fix_length(
    story_text,
    current_words,
    revision_number,
):

    if current_words < STORY_MIN_WORDS:

        direction = (
            "The manuscript is too short. "
            "Expand the SAME story without adding a new subplot. "
            "Add compatible scene detail, dialogue, interiority, "
            "physical action and transitions."
        )

    else:

        direction = (
            "The manuscript is too long. "
            "Tighten the SAME story without deleting any plot event, "
            "clue, motivation, chronology or ending. "
            "Remove redundancy and expendable description."
        )

    return writer_call(
        f"writer_length_revision_{revision_number}",

        (
            "<AUTHORITATIVE_OUTLINE>\n"
            f"{STORY_OUTLINE}\n"
            "</AUTHORITATIVE_OUTLINE>\n\n"

            "<CURRENT_MANUSCRIPT>\n"
            f"{story_text}\n"
            "</CURRENT_MANUSCRIPT>\n\n"

            f"Python counted the manuscript at exactly "
            f"{current_words} words.\n\n"

            f"{direction}\n\n"

            f"Return the COMPLETE revised manuscript at "
            f"{STORY_MIN_WORDS}-"
            f"{STORY_MAX_WORDS} words.\n"

            f"Target approximately "
            f"{STORY_TARGET_WORDS} words.\n"

            "Narration only."
        ),

        0.42,
    )


def story_is_usable(text):

    words = wc(
        text
    )

    complete = bool(
        re.search(
            r'[.!?…]["”»’\)\]]*$',
            text.strip(),
        )
    )

    return (
        STORY_MIN_WORDS
        <=
        words
        <=
        STORY_MAX_WORDS
        and
        complete
    )


print(
    "=" * 80
)

print(
    "AUDIO-STORY FACTORY — STRICT LOCAL MODEL PIPELINE"
)

print(
    "=" * 80
)

print(
    "Writer: Qwen/Qwen3-32B BF16"
)

print(
    "Judge : deepseek-ai/DeepSeek-R1-Distill-Qwen-32B BF16"
)

print(
    "Words :",
    f"{STORY_MIN_WORDS}-{STORY_MAX_WORDS}",
    "| target",
    STORY_TARGET_WORDS,
)


activate_writer()


(
    story,
    story_tokens,
    story_hit_cap,
) = writer_generate_initial()


save(
    Path(VERSIONS_DIR)
    /
    "writer_initial.txt",

    story,
)


print(
    "Initial writer words:",
    wc(
        story
    ),
    flush=True,
)


for revision_number in range(
    1,
    MAX_LENGTH_REVISIONS + 1,
):

    if story_is_usable(
        story
    ):
        break

    (
        story,
        story_tokens,
        story_hit_cap,
    ) = writer_fix_length(
        story,
        wc(
            story
        ),
        revision_number,
    )

    save(
        Path(VERSIONS_DIR)
        /
        f"writer_length_revision_{revision_number}.txt",

        story,
    )

    print(
        f"Length revision {revision_number}: "
        f"{wc(story)} words",
        flush=True,
    )


if not story_is_usable(
    story
):

    raise RuntimeError(
        "Writer failed the strict complete-story contract "
        "after length correction: "
        f"{wc(story)} words; "
        f"last generation hit token cap="
        f"{story_hit_cap}."
    )


# ============================================================
# PYTHON FREEZES + INDEXES NATURAL PARAGRAPHS
# ============================================================

story_words = wc(
    story
)

canonical = split_story(
    story
)

if len(
    canonical
) < 2:

    raise RuntimeError(
        "Writer did not produce usable "
        "natural paragraph boundaries."
    )


canonical_text = join_story(
    canonical
)

if wc(
    canonical_text
) != story_words:

    raise RuntimeError(
        "Paragraph indexing changed the word count."
    )


save(
    Path(VERSIONS_DIR)
    /
    "story_v0_canonical.txt",

    canonical_text,
)


save_json(
    Path(META_DIR)
    /
    "story_v0_index.json",

    {
        f"P{i:03d}":
            paragraph

        for i, paragraph
        in enumerate(
            canonical,
            1,
        )
    },
)


print(
    "Canonical frozen:",
    story_words,
    "words |",
    len(
        canonical
    ),
    "paragraphs",
    flush=True,
)


# ============================================================
# JUDGE
# ============================================================

JUDGE_SYSTEM = """
You are the logical-continuity JUDGE.

The AUTHORITATIVE OUTLINE outranks generated story text.

Audit exactly seven categories:

1. Outline fidelity.

2. Chronology.

3. Causality and motivation.

4. Character continuity, especially what each character knows.

5. Physical/spatial continuity.

6. Objects/clues/evidence and discovery timing.

7. Revelation/solution/ending.

Do NOT critique:

- style
- prose quality
- pacing
- atmosphere
- repetition
- aesthetics

Report only genuine material defects.

False positives are harmful.

For every genuine defect return:

category:
integer 1-7

paragraph:
exact target Pxxx

issue:
concise explanation

correction:
precise instruction to the writer

problem_anchor:
short EXACT substring from target paragraph

evidence_source:
OUTLINE or STORY

evidence_paragraph:
NONE for OUTLINE, otherwise exact Pxxx

evidence_anchor:
short EXACT substring from evidence source

must_remove:
exact bad substring from target, or NONE

must_contain:
exact substring copied from evidence_anchor that must appear,
or NONE

At least one of must_remove/must_contain must be non-NONE.

Use at most one flag per target paragraph.

Combine related defects.

For STORY-vs-STORY conflict, cite the earlier established
canonical fact.

If clean:

{"verdict":"PASS","flags":[]}

Otherwise:

{"verdict":"FLAG","flags":[{"category":1,"paragraph":"P007","issue":"...","correction":"...","problem_anchor":"...","evidence_source":"OUTLINE","evidence_paragraph":"NONE","evidence_anchor":"...","must_remove":"...","must_contain":"NONE"}]}

Reason internally.

Return one final JSON object only.
""".strip()


def validate_judge_flags(
    obj,
    paragraphs,
):

    if not isinstance(
        obj,
        dict,
    ):
        raise ValueError(
            "Judge output is not a JSON object."
        )

    verdict = str(
        obj.get(
            "verdict",
            "",
        )
    ).strip().upper()

    flags = obj.get(
        "flags",
        [],
    )

    if verdict not in {
        "PASS",
        "FLAG",
    }:
        raise ValueError(
            "Judge verdict must be PASS or FLAG."
        )

    if not isinstance(
        flags,
        list,
    ):
        raise ValueError(
            "Judge flags must be a list."
        )

    if (
        verdict == "PASS"
        and
        flags
    ):
        raise ValueError(
            "PASS cannot contain flags."
        )

    if (
        verdict == "FLAG"
        and
        not flags
    ):
        raise ValueError(
            "FLAG requires flags."
        )


    accepted = {}
    rejected = []


    for item_number, flag in enumerate(
        flags,
        1,
    ):

        reason = None


        if not isinstance(
            flag,
            dict,
        ):

            rejected.append({
                "item":
                    item_number,

                "paragraph":
                    None,

                "reason":
                    "malformed flag",
            })

            continue


        category = flag.get(
            "category"
        )

        target_p = pid(
            flag.get(
                "paragraph"
            )
        )

        issue = str(
            flag.get(
                "issue",
                "",
            )
        ).strip()

        correction = str(
            flag.get(
                "correction",
                "",
            )
        ).strip()


        if (
            not isinstance(
                category,
                int,
            )
            or
            not (
                1
                <=
                category
                <=
                7
            )
        ):

            reason = (
                "invalid category"
            )


        elif not (
            isinstance(
                target_p,
                int,
            )
            and
            1
            <=
            target_p
            <=
            len(
                paragraphs
            )
        ):

            reason = (
                "invalid target paragraph"
            )


        elif target_p in accepted:

            reason = (
                "duplicate target paragraph"
            )


        elif (
            not issue
            or
            not correction
        ):

            reason = (
                "missing issue/correction"
            )


        if reason is None:

            target_text = (
                paragraphs[
                    target_p - 1
                ]
            )

            problem_anchor = str(
                flag.get(
                    "problem_anchor",
                    "",
                )
            ).strip()

            if not occurs(
                problem_anchor,
                target_text,
            ):

                reason = (
                    "problem_anchor absent "
                    "from target paragraph"
                )


        if reason is None:

            evidence_source = str(
                flag.get(
                    "evidence_source",
                    "",
                )
            ).strip().upper()

            evidence_anchor = str(
                flag.get(
                    "evidence_anchor",
                    "",
                )
            ).strip()

            evidence_p = pid(
                flag.get(
                    "evidence_paragraph"
                ),
                allow_none=True,
            )


            if evidence_source == "OUTLINE":

                if evidence_p is not None:

                    reason = (
                        "OUTLINE evidence_paragraph "
                        "must be NONE"
                    )

                elif not occurs(
                    evidence_anchor,
                    STORY_OUTLINE,
                ):

                    reason = (
                        "evidence_anchor absent "
                        "from outline"
                    )


            elif evidence_source == "STORY":

                if not (
                    isinstance(
                        evidence_p,
                        int,
                    )
                    and
                    1
                    <=
                    evidence_p
                    <=
                    len(
                        paragraphs
                    )
                ):

                    reason = (
                        "invalid STORY evidence paragraph"
                    )

                elif not occurs(
                    evidence_anchor,
                    paragraphs[
                        evidence_p - 1
                    ],
                ):

                    reason = (
                        "evidence_anchor absent from "
                        "STORY evidence paragraph"
                    )

                elif evidence_p > target_p:

                    reason = (
                        "later story text cannot override "
                        "earlier canonical fact"
                    )

                elif (
                    evidence_p
                    ==
                    target_p
                    and
                    norm(
                        evidence_anchor
                    )
                    ==
                    norm(
                        problem_anchor
                    )
                ):

                    reason = (
                        "evidence merely repeats "
                        "problem anchor"
                    )


            else:

                reason = (
                    "evidence_source must be "
                    "OUTLINE or STORY"
                )


        if reason is None:

            must_remove = optional_text(
                flag.get(
                    "must_remove"
                )
            )

            must_contain = optional_text(
                flag.get(
                    "must_contain"
                )
            )


            if (
                must_remove is None
                and
                must_contain is None
            ):

                reason = (
                    "no mechanical repair contract"
                )


            elif (
                must_remove is not None
                and
                not occurs(
                    must_remove,
                    target_text,
                )
            ):

                reason = (
                    "must_remove absent from target"
                )


            elif (
                must_contain is not None
                and
                not occurs(
                    must_contain,
                    evidence_anchor,
                )
            ):

                reason = (
                    "must_contain not grounded "
                    "in evidence_anchor"
                )


        if reason is not None:

            rejected.append({
                "item":
                    item_number,

                "paragraph":
                    flag.get(
                        "paragraph"
                    ),

                "reason":
                    reason,
            })

            continue


        accepted[
            target_p
        ] = {
            "category":
                category,

            "paragraph":
                f"P{target_p:03d}",

            "issue":
                issue,

            "correction":
                correction,

            "problem_anchor":
                problem_anchor,

            "evidence_source":
                evidence_source,

            "evidence_paragraph":
                (
                    f"P{evidence_p:03d}"
                    if evidence_source
                    ==
                    "STORY"
                    else None
                ),

            "evidence_anchor":
                evidence_anchor,

            "must_remove":
                must_remove,

            "must_contain":
                must_contain,
        }


    return (
        accepted,
        rejected,
    )


def run_judge_audit(
    paragraphs,
    audit_number,
):

    indexed = numbered(
        paragraphs
    )

    save(
        Path(AUDITS_DIR)
        /
        f"audit_{audit_number}_input_story.txt",

        indexed,
    )


    (
        raw,
        _,
        _,
    ) = generate_local(
        f"judge_audit_{audit_number}",

        JUDGE_LLM,

        JUDGE_TOKENIZER,

        [
            {
                "role":
                    "system",

                "content":
                    JUDGE_SYSTEM,
            },

            {
                "role":
                    "user",

                "content":
                    (
                        "<AUTHORITATIVE_OUTLINE>\n"
                        f"{STORY_OUTLINE}\n"
                        "</AUTHORITATIVE_OUTLINE>\n\n"

                        "<CANONICAL_STORY>\n"
                        f"{indexed}\n"
                        "</CANONICAL_STORY>\n\n"

                        "Audit the complete story and "
                        "return the final JSON object."
                    ),
            },
        ],

        max_new_tokens=
            JUDGE_MAX_NEW_TOKENS,

        temperature=
            0.60,

        top_p=
            0.95,

        repetition_penalty=
            1.00,

        enable_thinking=
            None,
    )


    save(
        Path(AUDITS_DIR)
        /
        f"judge_audit_{audit_number}_raw.txt",

        raw,
    )


    obj = extract_json_dict(
        raw,
        required_keys=(
            "verdict",
            "flags",
        ),
    )


    (
        accepted,
        rejected,
    ) = validate_judge_flags(
        obj,
        paragraphs,
    )


    save_json(
        Path(AUDITS_DIR)
        /
        f"judge_audit_{audit_number}_parsed.json",

        obj,
    )


    save_json(
        Path(AUDITS_DIR)
        /
        f"judge_audit_{audit_number}_guardrail.json",

        {
            "accepted": {
                f"P{paragraph_number:03d}":
                    rule

                for paragraph_number, rule
                in sorted(
                    accepted.items()
                )
            },

            "rejected":
                rejected,
        },
    )


    print(
        "Judge verdict:",
        obj.get(
            "verdict"
        ),
        "| raw flags:",
        len(
            obj.get(
                "flags",
                [],
            )
        ),
        "| Python accepted:",
        len(
            accepted
        ),
        "| rejected:",
        len(
            rejected
        ),
        flush=True,
    )


    return {
        "audit":
            obj,

        "accepted":
            accepted,

        "rejected":
            rejected,

        "report":
            raw,
    }


# ============================================================
# FIRST JUDGE PASS
# ============================================================

activate_judge()


audit1 = run_judge_audit(
    canonical,
    1,
)


# ============================================================
# WRITER REPAIRS ONLY PYTHON-VALIDATED PARAGRAPHS
# ============================================================

REPAIR_SYSTEM = f"""
You are the fiction WRITER repairing specific paragraphs
of a frozen {STORY_LANGUAGE} manuscript.

Python owns the canonical manuscript.

You may change ONLY the paragraphs explicitly supplied.

Follow every correction contract exactly.

Preserve every unrelated fact, chronology, name and
paragraph role.

Return exactly one JSON object:

{{"replacements":[{{"paragraph":"P007","text":"complete replacement paragraph"}}]}}

Return all and only requested paragraph IDs.

No prose outside JSON.
""".strip()


def extract_replacements(
    raw,
    expected_ids,
):

    obj = extract_json_dict(
        raw,
        required_keys=(
            "replacements",
        ),
    )

    rows = obj.get(
        "replacements"
    )

    if not isinstance(
        rows,
        list,
    ):

        raise ValueError(
            "replacements must be a list"
        )


    replacements = {}


    for row in rows:

        if not isinstance(
            row,
            dict,
        ):
            continue

        paragraph_number = pid(
            row.get(
                "paragraph"
            )
        )

        text = str(
            row.get(
                "text",
                "",
            )
        ).strip()


        if (
            paragraph_number
            in expected_ids
            and
            paragraph_number
            not in replacements
            and
            text
        ):

            replacements[
                paragraph_number
            ] = re.sub(
                r"\s*\n\s*",
                " ",
                text,
            ).strip()


    return replacements


def repair_passes(
    frozen_paragraphs,
    accepted_flags,
):

    frozen = list(
        frozen_paragraphs
    )

    pending = set(
        accepted_flags
    )

    verified = {}

    current = {
        paragraph_number:
            frozen[
                paragraph_number
                -
                1
            ]

        for paragraph_number
        in pending
    }


    for pass_number in range(
        1,
        MAX_REPAIR_PASSES
        +
        1,
    ):

        if not pending:
            break


        contracts = []


        for paragraph_number in sorted(
            pending
        ):

            rule = accepted_flags[
                paragraph_number
            ]

            contracts.append({
                "paragraph":
                    f"P{paragraph_number:03d}",

                "current_text":
                    current[
                        paragraph_number
                    ],

                "issue":
                    rule[
                        "issue"
                    ],

                "correction":
                    rule[
                        "correction"
                    ],

                "evidence_source":
                    rule[
                        "evidence_source"
                    ],

                "evidence_paragraph":
                    (
                        rule[
                            "evidence_paragraph"
                        ]
                        or
                        "NONE"
                    ),

                "evidence_anchor":
                    rule[
                        "evidence_anchor"
                    ],

                "must_remove":
                    (
                        rule[
                            "must_remove"
                        ]
                        or
                        "NONE"
                    ),

                "must_contain":
                    (
                        rule[
                            "must_contain"
                        ]
                        or
                        "NONE"
                    ),
            })


        (
            raw,
            _,
            _,
        ) = generate_local(
            f"writer_repair_pass_{pass_number}",

            WRITER_LLM,

            WRITER_TOKENIZER,

            [
                {
                    "role":
                        "system",

                    "content":
                        REPAIR_SYSTEM,
                },

                {
                    "role":
                        "user",

                    "content":
                        (
                            "<AUTHORITATIVE_OUTLINE>\n"
                            f"{STORY_OUTLINE}\n"
                            "</AUTHORITATIVE_OUTLINE>\n\n"

                            "<AUTHORIZED_REPAIRS>\n"
                            f"{json.dumps(
                                contracts,
                                ensure_ascii=False,
                                indent=2,
                            )}\n"
                            "</AUTHORIZED_REPAIRS>\n\n"

                            "Correct only these paragraphs "
                            "and return the JSON object."
                        ),
                },
            ],

            max_new_tokens=
                REPAIR_MAX_NEW_TOKENS,

            temperature=
                0.35,

            top_p=
                0.90,

            repetition_penalty=
                1.02,

            enable_thinking=
                False,
        )


        save(
            Path(REVISIONS_DIR)
            /
            f"writer_repair_pass_{pass_number}_raw.txt",

            raw,
        )


        try:

            replacements = extract_replacements(
                raw,
                pending,
            )

        except Exception:

            replacements = {}


        next_pending = set()


        report = {
            "verified":
                {},

            "failed":
                {},
        }


        for paragraph_number in sorted(
            pending
        ):

            rule = accepted_flags[
                paragraph_number
            ]

            candidate = replacements.get(
                paragraph_number
            )

            reason = None


            if candidate is None:

                reason = (
                    "missing replacement"
                )


            elif (
                norm(
                    candidate
                )
                ==
                norm(
                    frozen[
                        paragraph_number
                        -
                        1
                    ]
                )
            ):

                reason = (
                    "paragraph unchanged"
                )


            elif (
                rule[
                    "must_remove"
                ]
                is not None
                and
                occurs(
                    rule[
                        "must_remove"
                    ],
                    candidate,
                )
            ):

                reason = (
                    "must_remove still present"
                )


            elif (
                rule[
                    "must_contain"
                ]
                is not None
                and
                not occurs(
                    rule[
                        "must_contain"
                    ],
                    candidate,
                )
            ):

                reason = (
                    "must_contain absent"
                )


            if reason is None:

                verified[
                    paragraph_number
                ] = candidate

                current[
                    paragraph_number
                ] = candidate

                report[
                    "verified"
                ][
                    f"P{paragraph_number:03d}"
                ] = candidate


            else:

                next_pending.add(
                    paragraph_number
                )

                if candidate:

                    current[
                        paragraph_number
                    ] = candidate

                report[
                    "failed"
                ][
                    f"P{paragraph_number:03d}"
                ] = reason


        save_json(
            Path(REVISIONS_DIR)
            /
            f"writer_repair_pass_{pass_number}_verification.json",

            report,
        )


        print(
            f"Repair pass {pass_number}: "
            f"verified "
            f"{len(pending - next_pending)} "
            f"| still pending "
            f"{len(next_pending)}",
            flush=True,
        )


        pending = next_pending


    if pending:

        raise RuntimeError(
            "Writer failed validated repairs: "
            +
            ", ".join(
                f"P{paragraph_number:03d}"
                for paragraph_number
                in sorted(
                    pending
                )
            )
        )


    revised = list(
        frozen
    )


    for (
        paragraph_number,
        replacement,
    ) in verified.items():

        revised[
            paragraph_number
            -
            1
        ] = replacement


    # Untouched paragraphs must remain byte-identical.

    for index, original in enumerate(
        frozen
    ):

        paragraph_number = (
            index
            +
            1
        )

        if (
            paragraph_number
            not in verified
            and
            revised[
                index
            ]
            !=
            original
        ):

            raise RuntimeError(
                "Unauthorized canonical change "
                f"to P{paragraph_number:03d}."
            )


    return (
        revised,
        verified,
    )


# ============================================================
# REPAIR + SPLICE + FINAL JUDGE
# ============================================================

if audit1[
    "accepted"
]:

    activate_writer()


    (
        canonical,
        verified_repairs,
    ) = repair_passes(
        canonical,
        audit1[
            "accepted"
        ],
    )


    revised_text = join_story(
        canonical
    )

    revised_words = wc(
        revised_text
    )


    if not (
        STORY_MIN_WORDS
        <=
        revised_words
        <=
        STORY_MAX_WORDS
    ):

        park_text_models()

        raise RuntimeError(
            "Repairs pushed story outside "
            "strict range: "
            f"{revised_words} words."
        )


    save(
        Path(VERSIONS_DIR)
        /
        "story_v1_repaired.txt",

        revised_text,
    )


    save_json(
        Path(REVISIONS_DIR)
        /
        "verified_replacements.json",

        {
            f"P{paragraph_number:03d}":
                text

            for paragraph_number, text
            in sorted(
                verified_repairs.items()
            )
        },
    )


    print(
        "Python splice complete:",
        revised_words,
        "words",
        flush=True,
    )


    activate_judge()


    final_audit = run_judge_audit(
        canonical,
        2,
    )


    if final_audit[
        "accepted"
    ]:

        save_json(
            Path(AUDITS_DIR)
            /
            "FINAL_VALIDATED_DEFECTS.json",

            {
                f"P{paragraph_number:03d}":
                    rule

                for paragraph_number, rule
                in sorted(
                    final_audit[
                        "accepted"
                    ].items()
                )
            },
        )


        park_text_models()


        raise RuntimeError(
            "Final Judge pass found "
            f"{len(final_audit['accepted'])} "
            "Python-validated defect(s). "
            "TTS will not run."
        )


else:

    verified_repairs = {}

    final_audit = audit1

    print(
        "No Python-validated defects; "
        "no repair pass needed.",
        flush=True,
    )


# ============================================================
# FINALIZE STORY
# ============================================================

FINAL_SCRIPT = join_story(
    canonical
)


FINAL_WORDS = wc(
    FINAL_SCRIPT
)


if not (
    STORY_MIN_WORDS
    <=
    FINAL_WORDS
    <=
    STORY_MAX_WORDS
):

    park_text_models()

    raise RuntimeError(
        "Final manuscript violates strict range: "
        f"{FINAL_WORDS} words."
    )


FINAL_STORY_PATH = (
    Path(STORY_DIR)
    /
    "final_narration_script.txt"
)


save(
    FINAL_STORY_PATH,
    FINAL_SCRIPT,
)


TTS_SCRIPT_PATH = (
    FINAL_STORY_PATH
)


STORYLINE_FIXED = True


park_text_models()


save_json(
    Path(META_DIR)
    /
    "story_pipeline_state.json",

    {
        "writer":
            "Qwen/Qwen3-32B",

        "judge":
            "deepseek-ai/"
            "DeepSeek-R1-Distill-Qwen-32B",

        "dtype":
            "bfloat16",

        "strict_word_min":
            STORY_MIN_WORDS,

        "strict_word_max":
            STORY_MAX_WORDS,

        "target_words":
            STORY_TARGET_WORDS,

        "story_min_new_tokens":
            STORY_MIN_NEW_TOKENS,

        "story_max_new_tokens":
            STORY_MAX_NEW_TOKENS,

        "final_words":
            FINAL_WORDS,

        "paragraphs":
            len(
                canonical
            ),

        "initial_judge_raw_flags":
            len(
                audit1[
                    "audit"
                ].get(
                    "flags",
                    [],
                )
            ),

        "initial_judge_python_accepted":
            len(
                audit1[
                    "accepted"
                ]
            ),

        "initial_judge_python_rejected":
            len(
                audit1[
                    "rejected"
                ]
            ),

        "writer_verified_repairs":
            len(
                verified_repairs
            ),

        "final_judge_python_accepted":
            len(
                final_audit[
                    "accepted"
                ]
            ),

        "storyline_fixed":
            True,

        "elapsed_minutes":
            round(
                (
                    time.perf_counter()
                    -
                    PIPELINE_START
                )
                /
                60.0,
                2,
            ),

        "final_story":
            str(
                FINAL_STORY_PATH
            ),
    },
)


print(
    "\n"
    +
    "=" * 80
)

print(
    "CELL 3 COMPLETE — STORYLINE FIXED"
)

print(
    "=" * 80
)

print(
    "Final words:",
    FINAL_WORDS,
)

print(
    "Paragraphs :",
    len(
        canonical
    ),
)

print(
    "Repairs    :",
    len(
        verified_repairs
    ),
)

print(
    "Writer device:",
    model_device(
        WRITER_LLM
    ),
)

print(
    "Judge device :",
    model_device(
        JUDGE_LLM
    ),
)

print(
    "TTS script:",
    TTS_SCRIPT_PATH,
)

AUDIO-STORY FACTORY — STRICT LOCAL MODEL PIPELINE
Writer: Qwen/Qwen3-32B BF16
Judge : deepseek-ai/DeepSeek-R1-Distill-Qwen-32B BF16
Words : 1300-1500 | target 1450
Swapping Writer: CPU -> A100 ...
Writer now on A100 (17.6s).
GPU residency: 64.36 / 79.25 GiB GPU used
writer_initial: input=521 tok | output=2131 tok | words=1450 | 184.7s
Initial writer words: 1450
Canonical frozen: 1450 words | 21 paragraphs
Swapping Writer: GPU -> CPU ...
Writer now in CPU RAM (42.5s).
Swapping Judge: CPU -> A100 ...
Judge now on A100 (18.8s).
GPU residency: 64.36 / 79.25 GiB GPU used
judge_audit_1: input=2941 tok | output=571 tok | words=407 | 39.4s
Judge verdict: PASS | raw flags: 0 | Python accepted: 0 | rejected: 0
No Python-validated defects; no repair pass needed.
Swapping Judge: GPU -> CPU ...
Judge now in CPU RAM (43.2s).

CELL 3 COMPLETE — STORYLINE FIXED
Final words: 1450
Paragraphs : 21
Repairs    : 0
Writer device: cpu
Judge device : cpu
TTS script: /content/audio_story_factory/runs/Romantic_

In [ ]:
# ============================================================
# CELL 4 — FAST BATCHED QWEN3-TTS + DOWNLOAD
# ============================================================

import gc
import hashlib
import json
import re
import time
import zipfile
from pathlib import Path

import numpy as np
import soundfile as sf
import torch
from google.colab import files as colab_files


# ------------------------------------------------------------
# A. VERIFY CELLS 1-3
# ------------------------------------------------------------

_required = (
    "TTS_ENGINE",
    "TTS_SCRIPT_PATH",
    "AUDIO_DIR",
    "AUDIO_CHUNKS_DIR",
    "RUN_ID",
    "STORYLINE_FIXED",
    "STORY_LANGUAGE",
)

_missing = [
    name
    for name in _required
    if name not in globals()
]

if _missing:
    raise RuntimeError(
        "Run Cells 1-3 first. Missing: "
        + ", ".join(_missing)
    )

if STORYLINE_FIXED is not True:
    raise RuntimeError(
        "Storyline is not fixed. TTS will not run."
    )

TTS_SCRIPT_PATH = Path(TTS_SCRIPT_PATH)
AUDIO_DIR = Path(AUDIO_DIR)
AUDIO_CHUNKS_DIR = Path(AUDIO_CHUNKS_DIR)

if not TTS_SCRIPT_PATH.is_file():
    raise FileNotFoundError(
        str(TTS_SCRIPT_PATH)
    )

AUDIO_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

AUDIO_CHUNKS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

script = TTS_SCRIPT_PATH.read_text(
    encoding="utf-8"
).strip()

if not script:
    raise RuntimeError(
        "Final narration script is empty."
    )

SCRIPT_SHA = hashlib.sha256(
    script.encode("utf-8")
).hexdigest()


# ------------------------------------------------------------
# B. MAKE SURE WRITER + JUDGE ARE OFF GPU
# ------------------------------------------------------------

def _first_device(model):
    return next(
        model.parameters()
    ).device


for _name in (
    "WRITER_LLM",
    "JUDGE_LLM",
):
    if _name not in globals():
        continue

    _model = globals()[_name]

    if _first_device(_model).type != "cpu":

        print(
            f"Moving {_name} to CPU...",
            flush=True,
        )

        _model.to("cpu")


gc.collect()
torch.cuda.empty_cache()

try:
    torch.cuda.ipc_collect()
except Exception:
    pass


# ------------------------------------------------------------
# C. MOVE ALREADY-LOADED QWEN3-TTS CPU -> GPU
# ------------------------------------------------------------

if not hasattr(
    TTS_ENGINE,
    "model",
):
    raise RuntimeError(
        "TTS_ENGINE is not the Qwen3TTSModel "
        "loaded in Cell 2."
    )

print(
    "Moving resident Qwen3-TTS to A100...",
    flush=True,
)

_t0 = time.perf_counter()

TTS_ENGINE.model.to(
    device="cuda:0",
    dtype=torch.bfloat16,
)

TTS_ENGINE.model.eval()

# Qwen3TTSModel uses this field to place inference inputs.
TTS_ENGINE.device = torch.device(
    "cuda:0"
)

torch.cuda.synchronize()

_free, _total = (
    torch.cuda.mem_get_info()
)

print(
    f"TTS ready in "
    f"{time.perf_counter() - _t0:.1f}s | "
    f"GPU used "
    f"{(_total - _free) / 1024**3:.2f} / "
    f"{_total / 1024**3:.2f} GiB",
    flush=True,
)


# ------------------------------------------------------------
# D. VOICE / LANGUAGE
#
# Language comes ONLY from Cell 1.
# Do not hard-code it here.
#
# Change only the speaker when desired.
# aiden / dylan / eric / ono_anna / ryan / serena / sohee / uncle_fu / vivian
# ------------------------------------------------------------

TTS_SPEAKER = "dylan"

TTS_LANGUAGE = STORY_LANGUAGE

TTS_INSTRUCT = (
    "Natural professional audiobook narration. "
    "Understand the story and respond naturally to changes in "
    "emotion, tension, dialogue and dramatic intensity. "
    "Use human pacing, pauses and emphasis. "
    "Remain cinematic and restrained rather than melodramatic."
)


_supported_speakers = (
    TTS_ENGINE.get_supported_speakers()
    or []
)

if _supported_speakers:

    _speaker_lookup = {
        str(name).lower():
            name

        for name
        in _supported_speakers
    }

    if (
        TTS_SPEAKER.lower()
        not in _speaker_lookup
    ):
        raise RuntimeError(
            f"Speaker {TTS_SPEAKER!r} unavailable. "
            f"Supported: {_supported_speakers}"
        )

    TTS_SPEAKER = (
        _speaker_lookup[
            TTS_SPEAKER.lower()
        ]
    )


_supported_languages = (
    TTS_ENGINE.get_supported_languages()
    or []
)

if _supported_languages:

    _language_lookup = {
        str(name).lower():
            name

        for name
        in _supported_languages
    }

    if (
        str(TTS_LANGUAGE).lower()
        not in _language_lookup
    ):
        raise RuntimeError(
            f"Story language "
            f"{TTS_LANGUAGE!r} is unavailable to Qwen3-TTS. "
            f"Supported: {_supported_languages}"
        )

    TTS_LANGUAGE = (
        _language_lookup[
            str(TTS_LANGUAGE).lower()
        ]
    )


print(
    "Speaker:",
    TTS_SPEAKER,
    "| Language inherited from Cell 1:",
    TTS_LANGUAGE,
    flush=True,
)


# ------------------------------------------------------------
# E. CHUNK STORY
# ------------------------------------------------------------

MIN_CHUNK_WORDS = 90
TARGET_CHUNK_WORDS = 140
MAX_CHUNK_WORDS = 175

TTS_BATCH_SIZE = 4


def tts_wc(text):
    return len(
        re.findall(
            r"\b[\w]+(?:['’\-][\w]+)*\b",
            str(text),
            flags=re.UNICODE,
        )
    )


def split_long_paragraph(text):

    text = text.strip()

    if (
        tts_wc(text)
        <=
        MAX_CHUNK_WORDS
    ):
        return [text]

    sentences = [
        sentence.strip()

        for sentence
        in re.split(
            r"(?<=[.!?…])\s+",
            text,
        )

        if sentence.strip()
    ]

    if len(sentences) <= 1:

        words = text.split()

        return [
            " ".join(
                words[
                    start:
                    start + MAX_CHUNK_WORDS
                ]
            )

            for start
            in range(
                0,
                len(words),
                MAX_CHUNK_WORDS,
            )
        ]

    pieces = []
    current = []

    for sentence in sentences:

        if (
            tts_wc(sentence)
            >
            MAX_CHUNK_WORDS
        ):

            if current:

                pieces.append(
                    " ".join(current)
                )

                current = []

            words = sentence.split()

            pieces.extend(
                " ".join(
                    words[
                        start:
                        start + MAX_CHUNK_WORDS
                    ]
                )

                for start
                in range(
                    0,
                    len(words),
                    MAX_CHUNK_WORDS,
                )
            )

            continue

        candidate = (
            " ".join(
                current + [sentence]
            )
            if current
            else sentence
        )

        if (
            current
            and
            tts_wc(candidate)
            >
            MAX_CHUNK_WORDS
        ):

            pieces.append(
                " ".join(current)
            )

            current = [sentence]

        else:

            current.append(
                sentence
            )

    if current:

        pieces.append(
            " ".join(current)
        )

    return pieces


def chunk_story(text):

    paragraphs = [
        paragraph.strip()

        for paragraph
        in re.split(
            r"\n\s*\n",
            text.strip(),
        )

        if paragraph.strip()
    ]

    units = []

    for paragraph in paragraphs:

        units.extend(
            split_long_paragraph(
                paragraph
            )
        )

    chunks = []
    current = []

    for unit in units:

        if not current:

            current = [unit]

            continue

        current_text = (
            "\n\n".join(current)
        )

        candidate = (
            "\n\n".join(
                current + [unit]
            )
        )

        if (
            tts_wc(candidate)
            >
            TARGET_CHUNK_WORDS
            and
            tts_wc(current_text)
            >=
            MIN_CHUNK_WORDS
        ):

            chunks.append(
                current_text
            )

            current = [unit]

        elif (
            tts_wc(candidate)
            >
            MAX_CHUNK_WORDS
        ):

            chunks.append(
                current_text
            )

            current = [unit]

        else:

            current.append(
                unit
            )

    if current:

        chunks.append(
            "\n\n".join(current)
        )

    if (
        len(chunks) >= 2
        and
        tts_wc(chunks[-1])
        <
        MIN_CHUNK_WORDS
        and
        tts_wc(
            chunks[-2]
            + "\n\n"
            + chunks[-1]
        )
        <=
        MAX_CHUNK_WORDS
    ):

        chunks[-2] = (
            chunks[-2]
            + "\n\n"
            + chunks[-1]
        )

        chunks.pop()

    return chunks


chunks = chunk_story(
    script
)

if not chunks:
    raise RuntimeError(
        "TTS chunker produced no chunks."
    )

print(
    "TTS chunks:",
    len(chunks),
    "| batch size:",
    TTS_BATCH_SIZE,
    "| word counts:",
    [
        tts_wc(chunk)
        for chunk in chunks
    ],
    flush=True,
)


# ------------------------------------------------------------
# F. CHECKPOINT MANIFEST
# ------------------------------------------------------------

MANIFEST_PATH = (
    AUDIO_DIR
    / "tts_manifest.json"
)

_manifest_identity = {
    "script_sha256":
        SCRIPT_SHA,

    "speaker":
        str(TTS_SPEAKER),

    "language":
        str(TTS_LANGUAGE),

    "instruct":
        TTS_INSTRUCT,

    "chunks":
        chunks,

    "generation": {
        "do_sample":
            True,

        "top_p":
            0.95,

        "temperature":
            0.85,

        "repetition_penalty":
            1.05,

        "max_new_tokens":
            2048,
    },
}

_reuse_chunks = False

if MANIFEST_PATH.exists():

    try:

        _old_manifest = json.loads(
            MANIFEST_PATH.read_text(
                encoding="utf-8"
            )
        )

        _reuse_chunks = (
            _old_manifest
            ==
            _manifest_identity
        )

    except Exception:

        _reuse_chunks = False


if not _reuse_chunks:

    for _old_wav in (
        AUDIO_CHUNKS_DIR.glob(
            "chunk_*.wav"
        )
    ):

        _old_wav.unlink()

    MANIFEST_PATH.write_text(
        json.dumps(
            _manifest_identity,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )


# ------------------------------------------------------------
# G. FIND CACHED / MISSING CHUNKS
# ------------------------------------------------------------

sample_rate = None
missing_indices = []

for index in range(
    1,
    len(chunks) + 1,
):

    chunk_path = (
        AUDIO_CHUNKS_DIR
        /
        f"chunk_{index:03d}.wav"
    )

    if (
        _reuse_chunks
        and
        chunk_path.is_file()
    ):

        _cached_wav, _cached_sr = (
            sf.read(
                chunk_path,
                dtype="float32",
            )
        )

        _cached_sr = int(
            _cached_sr
        )

        if sample_rate is None:

            sample_rate = (
                _cached_sr
            )

        elif (
            _cached_sr
            !=
            sample_rate
        ):

            raise RuntimeError(
                "Cached TTS chunk sample rates disagree."
            )

        print(
            f"TTS {index}/{len(chunks)}: cached",
            flush=True,
        )

    else:

        missing_indices.append(
            index
        )


# ------------------------------------------------------------
# H. BATCHED QWEN3-TTS
# ------------------------------------------------------------

batch_size = min(
    TTS_BATCH_SIZE,
    max(
        1,
        len(missing_indices),
    ),
)

position = 0

_tts_started = (
    time.perf_counter()
)

while (
    position
    <
    len(missing_indices)
):

    batch_indices = (
        missing_indices[
            position:
            position + batch_size
        ]
    )

    batch_texts = [
        chunks[
            index - 1
        ]

        for index
        in batch_indices
    ]

    batch_count = len(
        batch_texts
    )

    print(
        "TTS batch:",
        ", ".join(
            f"{index}/{len(chunks)}"
            for index
            in batch_indices
        ),
        "| words:",
        [
            tts_wc(text)
            for text in batch_texts
        ],
        flush=True,
    )

    try:

        torch.cuda.synchronize()

        _batch_started = (
            time.perf_counter()
        )

        wavs, sr = (
            TTS_ENGINE
            .generate_custom_voice(
                text=
                    batch_texts,

                speaker=
                    [
                        TTS_SPEAKER
                    ]
                    *
                    batch_count,

                language=
                    [
                        TTS_LANGUAGE
                    ]
                    *
                    batch_count,

                instruct=
                    [
                        TTS_INSTRUCT
                    ]
                    *
                    batch_count,

                non_streaming_mode=
                    True,

                do_sample=
                    True,

                top_p=
                    0.95,

                temperature=
                    0.85,

                repetition_penalty=
                    1.05,

                max_new_tokens=
                    2048,
            )
        )

        torch.cuda.synchronize()

    except torch.cuda.OutOfMemoryError:

        gc.collect()
        torch.cuda.empty_cache()

        if batch_size == 1:
            raise

        batch_size = max(
            1,
            batch_size // 2,
        )

        print(
            "CUDA OOM; retrying same batch "
            f"with batch size {batch_size}.",
            flush=True,
        )

        continue


    print(
        f"Batch completed in "
        f"{time.perf_counter() - _batch_started:.1f}s",
        flush=True,
    )


    if (
        not isinstance(
            wavs,
            (list, tuple),
        )
        or
        len(wavs)
        !=
        len(batch_indices)
    ):

        raise RuntimeError(
            "Qwen3-TTS returned the wrong "
            "number of audio clips."
        )


    sr = int(sr)

    if sample_rate is None:

        sample_rate = sr

    elif sr != sample_rate:

        raise RuntimeError(
            "TTS sample rate changed between batches."
        )


    for index, wav_data in zip(
        batch_indices,
        wavs,
    ):

        if wav_data is None:

            raise RuntimeError(
                f"TTS returned no audio "
                f"for chunk {index}."
            )

        wav = np.asarray(
            wav_data,
            dtype=np.float32,
        ).squeeze()

        if wav.size == 0:

            raise RuntimeError(
                f"TTS returned empty audio "
                f"for chunk {index}."
            )

        sf.write(
            AUDIO_CHUNKS_DIR
            /
            f"chunk_{index:03d}.wav",

            wav,

            sample_rate,
        )


    position += len(
        batch_indices
    )


if sample_rate is None:
    raise RuntimeError(
        "No TTS sample rate was established."
    )


print(
    f"All TTS chunks ready in "
    f"{(time.perf_counter() - _tts_started) / 60:.2f} min.",
    flush=True,
)


# ------------------------------------------------------------
# I. CONCATENATE IN ORIGINAL ORDER
# ------------------------------------------------------------

audio_parts = []

silence = np.zeros(
    int(
        sample_rate * 0.12
    ),
    dtype=np.float32,
)

for index in range(
    1,
    len(chunks) + 1,
):

    chunk_path = (
        AUDIO_CHUNKS_DIR
        /
        f"chunk_{index:03d}.wav"
    )

    if not chunk_path.is_file():

        raise RuntimeError(
            f"Missing TTS chunk: "
            f"{chunk_path}"
        )

    wav, sr = sf.read(
        chunk_path,
        dtype="float32",
    )

    wav = np.asarray(
        wav,
        dtype=np.float32,
    ).squeeze()

    if int(sr) != sample_rate:

        raise RuntimeError(
            f"Sample-rate mismatch in "
            f"{chunk_path.name}."
        )

    audio_parts.append(
        wav
    )

    if index < len(chunks):

        audio_parts.append(
            silence
        )


final_audio = np.concatenate(
    audio_parts
)


_safe_language = re.sub(
    r"[^A-Za-z0-9_-]+",
    "_",
    str(TTS_LANGUAGE).lower(),
).strip("_")


FINAL_AUDIO_PATH = (
    AUDIO_DIR
    /
    f"{RUN_ID}_audiobook_{_safe_language}.wav"
)


sf.write(
    FINAL_AUDIO_PATH,
    final_audio,
    sample_rate,
)


duration_seconds = (
    len(final_audio)
    /
    sample_rate
)


# ------------------------------------------------------------
# J. RETURN TTS TO CPU RAM
# ------------------------------------------------------------

print(
    "Returning Qwen3-TTS to CPU RAM...",
    flush=True,
)

TTS_ENGINE.model.to(
    device="cpu",
    dtype=torch.bfloat16,
)

TTS_ENGINE.device = (
    torch.device("cpu")
)

gc.collect()
torch.cuda.empty_cache()

try:
    torch.cuda.ipc_collect()
except Exception:
    pass


# ------------------------------------------------------------
# K. METADATA
# ------------------------------------------------------------

TTS_META_PATH = (
    AUDIO_DIR
    /
    "tts_output.json"
)


TTS_META_PATH.write_text(
    json.dumps(
        {
            "run_id":
                RUN_ID,

            "story_language":
                str(STORY_LANGUAGE),

            "tts_language":
                str(TTS_LANGUAGE),

            "script_sha256":
                SCRIPT_SHA,

            "speaker":
                str(TTS_SPEAKER),

            "requested_batch_size":
                TTS_BATCH_SIZE,

            "effective_batch_size":
                batch_size,

            "sample_rate":
                sample_rate,

            "chunks":
                len(chunks),

            "duration_seconds":
                duration_seconds,

            "output":
                str(FINAL_AUDIO_PATH),
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# L. PACKAGE + DOWNLOAD
# ------------------------------------------------------------

DOWNLOAD_ZIP = (
    Path("/content")
    /
    f"{RUN_ID}_deliverables.zip"
)


with zipfile.ZipFile(
    DOWNLOAD_ZIP,
    mode="w",
    compression=zipfile.ZIP_STORED,
) as zf:

    zf.write(
        TTS_SCRIPT_PATH,
        arcname=
            "final_narration_script.txt",
    )

    zf.write(
        FINAL_AUDIO_PATH,
        arcname=
            FINAL_AUDIO_PATH.name,
    )

    zf.write(
        TTS_META_PATH,
        arcname=
            "tts_output.json",
    )


print(
    "\n"
    + "=" * 76
)

print(
    "CELL 4 COMPLETE — BATCHED QWEN3-TTS"
)

print(
    "=" * 76
)

print(
    "Story language:",
    STORY_LANGUAGE,
)

print(
    "TTS language:",
    TTS_LANGUAGE,
)

print(
    "Speaker:",
    TTS_SPEAKER,
)

print(
    "Audio duration:",
    f"{duration_seconds / 60:.2f} min",
)

print(
    "Audio:",
    FINAL_AUDIO_PATH,
)

print(
    "Download:",
    DOWNLOAD_ZIP,
)


colab_files.download(
    str(DOWNLOAD_ZIP)
)

Moving resident Qwen3-TTS to A100...
TTS ready in 0.0s | GPU used 6.80 / 79.25 GiB
Speaker: dylan | Language inherited from Cell 1: english
TTS chunks: 12 | batch size: 4 | word counts: [103, 73, 110, 139, 105, 138, 98, 168, 111, 126, 131, 154]
TTS batch: 1/12, 2/12, 3/12, 4/12 | words: [103, 73, 110, 139]


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


Batch completed in 282.6s
TTS batch: 5/12, 6/12, 7/12, 8/12 | words: [105, 138, 98, 168]


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


Batch completed in 341.6s
TTS batch: 9/12, 10/12, 11/12, 12/12 | words: [111, 126, 131, 154]


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
